# ED Admission Prediction — Model Development Report

Sprint 2 (Machine Learning Model Development). Presentation layer over already-computed results — every table and chart below loads saved artifacts (`ML/reports/modeling/experiment_log.json`, `model_comparison.csv`, etc.); **no model is retrained or re-tuned by running this notebook.**

Full markdown reports: `ML/reports/modeling/`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import pandas as pd
from IPython.display import Image, display

MODELING_DIR = REPO_ROOT / "ML" / "reports" / "modeling"
FIGURES_DIR = MODELING_DIR / "figures"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

## 1. Models Trained

7 candidate models (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost, LightGBM, CatBoost) plus a stacking ensemble, each tuned with `RandomizedSearchCV` (15 iterations, 3-fold, scored on ROC-AUC) and evaluated with 5-fold cross-validation on the training split.

In [2]:
experiment_log = json.loads((MODELING_DIR / "experiment_log.json").read_text())
print(f"{len(experiment_log)} models in the experiment log")
for record in experiment_log:
    print(f"  {record['model_name']:22s} trained in {record['training_time_seconds']:.1f}s")

8 models in the experiment log
  logistic_regression    trained in 146.1s
  decision_tree          trained in 25.0s
  random_forest          trained in 155.1s
  gradient_boosting      trained in 1219.8s
  xgboost                trained in 109.6s
  lightgbm               trained in 74.9s
  catboost               trained in 485.3s
  stacking_ensemble      trained in 109.4s


### Hyperparameter Tuning Results

Best parameters found per model (from `RandomizedSearchCV`).

In [3]:
for record in experiment_log:
    print(f"{record['model_name']}:")
    print(f"  {record['hyperparameters']}")

logistic_regression:
  {'penalty': 'l2', 'class_weight': 'balanced', 'C': 0.1}
decision_tree:
  {'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 5, 'class_weight': 'balanced'}
random_forest:
  {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.3, 'max_depth': 20, 'class_weight': 'balanced_subsample'}
gradient_boosting:
  {'subsample': 0.7, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05}
xgboost:
  {'subsample': 1.0, 'scale_pos_weight': 1, 'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
lightgbm:
  {'subsample': 0.85, 'num_leaves': 127, 'n_estimators': 400, 'max_depth': -1, 'learning_rate': 0.05, 'class_weight': 'balanced'}
catboost:
  {'learning_rate': 0.1, 'l2_leaf_reg': 10, 'iterations': 600, 'depth': 4, 'auto_class_weights': 'Balanced'}
stacking_ensemble:
  {'base_models': ['lightgbm', 'xgboost', 'random_forest'], 'final_estimator': 'LogisticRegression'}


### Cross-Validation Results (5-fold, training split)

In [4]:
cv_rows = []
for record in experiment_log:
    agg = record["cross_validation"].get("aggregated", {}).get("roc_auc", {})
    cv_rows.append({
        "model": record["model_name"],
        "cv_roc_auc_mean": agg.get("mean"),
        "cv_roc_auc_std": agg.get("std"),
        "n_folds": record["cross_validation"].get("n_folds"),
    })
cv_table = pd.DataFrame(cv_rows).sort_values("cv_roc_auc_mean", ascending=False)
cv_table

,model,cv_roc_auc_mean,cv_roc_auc_std,n_folds
5,lightgbm,0.951803,0.007059,5
4,xgboost,0.950105,0.007625,5
6,catboost,0.945256,0.007426,5
3,gradient_boosting,0.941247,0.007641,5
2,random_forest,0.938305,0.005646,5
0,logistic_regression,0.924309,0.007739,5
1,decision_tree,0.875886,0.006744,5
7,stacking_ensemble,NaN,NaN,0


## 2. Evaluation Metrics — All Models (validation split)

Full metric suite: accuracy, precision, recall, specificity, F1, ROC-AUC, PR-AUC, Brier score, training time, and interpretability tier.

In [5]:
model_comparison = pd.read_csv(MODELING_DIR / "model_comparison.csv")
model_comparison.round(4)

,model,interpretability,accuracy,precision,recall,specificity,f1,roc_auc,pr_auc,brier_score,cv_roc_auc_mean,cv_roc_auc_std,training_time_seconds
0,lightgbm,medium,0.9330,0.7716,0.7013,0.9684,0.7348,0.9649,0.8275,0.0528,0.9518,0.0071,74.922
1,catboost,medium,0.9089,0.6098,0.8648,0.9156,0.7152,0.9609,0.8135,0.0668,0.9453,0.0074,485.312
2,xgboost,medium,0.9305,0.7871,0.6509,0.9732,0.7126,0.9607,0.8106,0.0516,0.9501,0.0076,109.625
3,stacking_ensemble,low (meta-ensemble),0.9326,0.8023,0.6509,0.9756,0.7188,0.9552,0.7874,0.0529,NaN,NaN,109.437
4,gradient_boosting,medium,0.9255,0.7565,0.6447,0.9684,0.6961,0.9515,0.7669,0.0557,0.9412,0.0076,1219.844
5,random_forest,medium,0.9222,0.7835,0.5692,0.9760,0.6594,0.9492,0.7550,0.0581,0.9383,0.0056,155.109
6,logistic_regression,high,0.8669,0.4982,0.8585,0.8682,0.6305,0.9387,0.7187,0.0984,0.9243,0.0077,146.078
7,decision_tree,high,0.8303,0.4269,0.8270,0.8308,0.5632,0.8930,0.5857,0.1259,0.8759,0.0067,25.031


## 3. Model Comparison — Visualizations

![ROC Curves](../reports/modeling/figures/roc_curves.png)

![PR Curves](../reports/modeling/figures/pr_curves.png)

![Calibration Curves](../reports/modeling/figures/calibration_curves.png)

![Metric Comparison](../reports/modeling/figures/metric_comparison_bars.png)

## 4. Survey-Aware Model Comparison

Project research objective: does incorporating NHAMCS's survey design (weights/strata/clusters) change the model's conclusions? Full report: `ML/reports/modeling/survey_aware_comparison.md`.

In [6]:
glm_comparison = pd.read_csv(MODELING_DIR / "survey_aware_glm_comparison.csv", index_col=0)
glm_comparison.round(4)

,coefficient,weighted_std_error,weighted_p_value,significant_unweighted,cluster_robust_std_error,cluster_robust_p_value,significant_cluster_robust
const,-3.8028,0.0011,0.0,True,0.2924,0.0000,True
TOTDIAG,0.2939,0.0005,0.0,True,0.1415,0.0378,True
CONSULT__Yes,1.8170,0.0008,0.0,True,0.2316,0.0000,True
NUMDIS,-0.2410,0.0005,0.0,True,0.1966,0.2202,False
NUMGIV,0.2900,0.0006,0.0,True,0.1085,0.0075,True
CBC__Yes,0.4456,0.0013,0.0,True,0.3006,0.1382,False
LOV,0.1708,0.0003,0.0,True,0.0763,0.0252,True
IVFLUIDS__Yes,0.5185,0.0009,0.0,True,0.1655,0.0017,True
DIAG1__frequency,-16.1264,0.0510,0.0,True,8.4715,0.0570,False
AGE,0.5537,0.0004,0.0,True,0.0840,0.0000,True


**Key finding**: 8 of 15 predictors are statistically significant (p<0.05) under naive weighted standard errors but lose significance once cluster-robust correction is applied — the naive model overstated confidence by ignoring the clustered (by hospital/PSU) sample design.

## 5. Final Model Selection

Selection criteria: primary = validation PR-AUC (more informative than ROC-AUC at ~13% positive rate); tie-breakers = recall, then training time/interpretability.

In [7]:
best_model_name = model_comparison.sort_values("pr_auc", ascending=False).iloc[0]["model"]
best_record = next(r for r in experiment_log if r["model_name"] == best_model_name)
print(f"Selected model: {best_model_name}")
print(f"Validation metrics: {best_record['validation_metrics']}")

Selected model: lightgbm
Validation metrics: {'accuracy': 0.9330282861896838, 'precision': 0.7716262975778547, 'recall': 0.7012578616352201, 'sensitivity': 0.7012578616352201, 'specificity': 0.9683604985618408, 'f1': 0.7347611202635914, 'roc_auc': 0.9648510284194721, 'pr_auc': 0.8274761599325219, 'brier_score': 0.052812447645099125, 'confusion_matrix': {'true_negative': 2020, 'false_positive': 66, 'false_negative': 95, 'true_positive': 223}, 'threshold': 0.5}


## 6. Key Findings

- **LightGBM** selected: highest validation PR-AUC (0.8275), among the fastest models to train (75s), test-split ROC-AUC 0.9564 confirms good generalization (evaluated exactly once, after selection).
- The stacking ensemble (LightGBM+XGBoost+Random Forest) did **not** beat LightGBM alone — the three base learners are all tree ensembles whose predictions correlate heavily, leaving the meta-learner little complementary signal.
- Gradient Boosting (sklearn) was the slowest model by a wide margin (1220s vs. 75-485s for the other boosted-tree models) due to no internal parallelism — a practical, not just predictive, consideration when choosing a production model.

## Next Steps

- Full comparison report: `ML/reports/modeling/model_comparison.md`
- Final selection rationale: `ML/reports/modeling/final_model_selection.md`
- Explainability analysis: `ML/notebooks/explainability_report.ipynb`